In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl

# Select rows where ISSUE contains "Attribution" (case-sensitive)
df_attr_concat = df[
    df["ISSUE"].str.contains("Attribution", na=False) &
    df["ISSUE"].str.contains("Concatenate", na=False)
]

df_attr_concat =  df_attr_concat[['ISSUE', 'Station ID', 'Unique Names', 'Native ID']].reset_index(drop=True)


df_attr_concat["Station ID"] = pd.to_numeric(df_attr_concat["Station ID"], errors="coerce").astype("Int64")

len(df_attr_concat)

df_attr_concat

,ISSUE,Station ID,Unique Names,Native ID
0,"Attribution, Concatenate",12559,NA -> Peace Valley Attachie Flat Upper Terrace,Peace Valley Attachie Flat Upper Terrace -> AFU
1,"Attribution, Concatenate and delete",12592,NaN,Burnaby Kensington Park -> T004
2,"Attribution, Concatenate and delete",12611,NaN,North Vancouver Second Narrows -> T006
3,"Attribution, Concatenate and delete",12563,NA -> Rocky Point Park,Port Moody Rocky Point Park -> T009
4,"Attribution, ID, Name, Location, Concatenate w...",12582,NA -> Vancouver Clark Drive,Vancouver Clark Drive -> T050
5,"Attribution, ID, Name, Location, Concatenate w...",12949,Vancouver Clark Drive,E316370 -> T050
6,"Attribution,Concatenate w/12536",2641,Chilliwack Airport,M110517 -> T012
7,"Attribution,Concatenate w/12536",2637,Chilliwack Airport Met,M110516 -> T012


In [5]:
import re

def parse_native_id(s):
    if pd.isna(s):
        return pd.Series([None, None])

    s = str(s)  # <-- KEY FIX

    parts = re.split(r'\s*-?\?>\s*|\s*->\s*', s)

    old_id = parts[0].strip()
    new_id = parts[-1].strip()

    return pd.Series([old_id, new_id])

df_attr_concat[['old_native_id', 'new_native_id']] = df_attr_concat['Native ID'].apply(parse_native_id)

df_attr_concat = df_attr_concat.drop(columns= ['Native ID'])

df_attr_concat


,ISSUE,Station ID,Unique Names,old_native_id,new_native_id
0,"Attribution, Concatenate",12559,NA -> Peace Valley Attachie Flat Upper Terrace,Peace Valley Attachie Flat Upper Terrace,AFU
1,"Attribution, Concatenate and delete",12592,NaN,Burnaby Kensington Park,T004
2,"Attribution, Concatenate and delete",12611,NaN,North Vancouver Second Narrows,T006
3,"Attribution, Concatenate and delete",12563,NA -> Rocky Point Park,Port Moody Rocky Point Park,T009
4,"Attribution, ID, Name, Location, Concatenate w...",12582,NA -> Vancouver Clark Drive,Vancouver Clark Drive,T050
5,"Attribution, ID, Name, Location, Concatenate w...",12949,Vancouver Clark Drive,E316370,T050
6,"Attribution,Concatenate w/12536",2641,Chilliwack Airport,M110517,T012
7,"Attribution,Concatenate w/12536",2637,Chilliwack Airport Met,M110516,T012


In [8]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s
     ) t) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN meta_station s_old ON m_old.station_id = s_old.station_id
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN meta_station s_new ON m_new.station_id = s_new.station_id
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE s_old.native_id = %s
       AND s_new.native_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;

"""


def count_station_stats(old_id, new_id, engine):

    params = (
    old_id,          # n_old
    new_id,          # n_new
    old_id, new_id,  # n_overlap
    old_id, new_id   # n_overlap_same_datum
    )

    df = pd.read_sql(
        query,
        engine,
        params = params
    )
    return df.iloc[0]

# df_test = df_concat_new.head(3).copy()


stats = df_attr_concat.apply(
    lambda r: count_station_stats(r['old_native_id'], r['new_native_id'], engine),
    axis=1
)

df_attr_concat[['n_old', 'n_new', 'n_overlap', 'n_overlap_same_datum']] = stats

In [49]:
df_attr_concat

,ISSUE,Station ID,Unique Names,old_native_id,new_native_id,n_old,n_new,n_overlap,n_overlap_same_datum,old_native_id_db,new_station_status
0,"Attribution, Concatenate",12559,NA -> Peace Valley Attachie Flat Upper Terrace,Peace Valley Attachie Flat Upper Terrace,AFU,0,479390,0,0,Peace Valley Attachie Flat Upper Terrace,EXISTS
1,"Attribution, Concatenate and delete",12592,NaN,Burnaby Kensington Park,T004,574,91903,574,574,Burnaby Kensington Park,EXISTS
2,"Attribution, Concatenate and delete",12611,NaN,North Vancouver Second Narrows,T006,574,69550,574,574,North Vancouver Second Narrows,EXISTS
3,"Attribution, Concatenate and delete",12563,NA -> Rocky Point Park,Port Moody Rocky Point Park,T009,1148,156449,1148,1148,Port Moody Rocky Point Park,EXISTS
4,"Attribution, ID, Name, Location, Concatenate w...",12582,NA -> Vancouver Clark Drive,Vancouver Clark Drive,T050,1148,0,0,0,Vancouver Clark Drive,NOT EXISTS
5,"Attribution, ID, Name, Location, Concatenate w...",12949,Vancouver Clark Drive,E316370,T050,103270,0,0,0,E316370,NOT EXISTS
6,"Attribution,Concatenate w/12536",2641,Chilliwack Airport,M110517,T012,0,160146,0,0,M110517,EXISTS
7,"Attribution,Concatenate w/12536",2637,Chilliwack Airport Met,M110516,T012,0,160146,0,0,M110516,EXISTS


### Reselect by station_id, has checked that the old station_id and native_id are the same

In [48]:
query = """
SELECT
    -- old station native_id (from DB)
    (SELECT s.native_id
     FROM meta_station s
     WHERE s.station_id = %s
    ) AS old_native_id_db,

    -- old count (by old station_id)
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s) AS n_old,

    -- new count (by new native_id)
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s
     ) t) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN meta_station s_new ON m_new.station_id = s_new.station_id
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE m_old.station_id = %s
       AND s_new.native_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def count_station_stats(old_station_id, new_native_id, engine):

    params = (
        int(old_station_id),      # old_native_id_db
        int(old_station_id),      # n_old
        str(new_native_id),       # n_new
        int(old_station_id),      # n_overlap (old)
        str(new_native_id),       # n_overlap (new)
        int(old_station_id),      # n_overlap_same_datum (old)
        str(new_native_id)        # n_overlap_same_datum (new)
    )

    df = pd.read_sql(
        query,
        engine,
        params=params
    )

    return df.iloc[0]


stats = df_attr_concat.apply(
    lambda r: count_station_stats(
        r['Station ID'],
        r['new_native_id'],
        engine
    ),
    axis=1
)

df_attr_concat[
    [
        'old_native_id_db',
        'n_old',
        'n_new',
        'n_overlap',
        'n_overlap_same_datum'
    ]
] = stats

df_attr_concat

,ISSUE,Station ID,Unique Names,old_native_id,new_native_id,n_old,n_new,n_overlap,n_overlap_same_datum,old_native_id_db,new_station_status
0,"Attribution, Concatenate",12559,NA -> Peace Valley Attachie Flat Upper Terrace,Peace Valley Attachie Flat Upper Terrace,AFU,0,479390,0,0,Peace Valley Attachie Flat Upper Terrace,EXISTS
1,"Attribution, Concatenate and delete",12592,NaN,Burnaby Kensington Park,T004,574,91903,574,574,Burnaby Kensington Park,EXISTS
2,"Attribution, Concatenate and delete",12611,NaN,North Vancouver Second Narrows,T006,574,69550,574,574,North Vancouver Second Narrows,EXISTS
3,"Attribution, Concatenate and delete",12563,NA -> Rocky Point Park,Port Moody Rocky Point Park,T009,1148,156449,1148,1148,Port Moody Rocky Point Park,EXISTS
4,"Attribution, ID, Name, Location, Concatenate w...",12582,NA -> Vancouver Clark Drive,Vancouver Clark Drive,T050,1148,0,0,0,Vancouver Clark Drive,NOT EXISTS
5,"Attribution, ID, Name, Location, Concatenate w...",12949,Vancouver Clark Drive,E316370,T050,103270,0,0,0,E316370,NOT EXISTS
6,"Attribution,Concatenate w/12536",2641,Chilliwack Airport,M110517,T012,0,160146,0,0,M110517,EXISTS
7,"Attribution,Concatenate w/12536",2637,Chilliwack Airport Met,M110516,T012,0,160146,0,0,M110516,EXISTS


In [12]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.native_id = :native_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in df_attr_concat.iterrows():
        native_id = row['new_native_id']

        exists = conn.execute(
            exists_sql,
            {"native_id": native_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(df_attr_concat)}] "
            f"new_native_id={native_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
df_attr_concat['new_station_status'] = exists_list

[  1/8] new_native_id=AFU             | → EXISTS
[  2/8] new_native_id=T004            | → EXISTS
[  3/8] new_native_id=T006            | → EXISTS
[  4/8] new_native_id=T009            | → EXISTS
[  5/8] new_native_id=T050            | → NOT EXISTS
[  6/8] new_native_id=T050            | → NOT EXISTS
[  7/8] new_native_id=T012            | → EXISTS
[  8/8] new_native_id=T012            | → EXISTS


### First 4, using native_id to concat

In [16]:
df_attr_concat_native = df_attr_concat[0:4]

df_attr_concat_native

,ISSUE,Station ID,Unique Names,old_native_id,new_native_id,n_old,n_new,n_overlap,n_overlap_same_datum,old_native_id_db,new_station_status
0,"Attribution, Concatenate",12559,NA -> Peace Valley Attachie Flat Upper Terrace,Peace Valley Attachie Flat Upper Terrace,AFU,2819,476571,0,0,Peace Valley Attachie Flat Upper Terrace,EXISTS
1,"Attribution, Concatenate and delete",12592,NaN,Burnaby Kensington Park,T004,2725,89752,574,574,Burnaby Kensington Park,EXISTS
2,"Attribution, Concatenate and delete",12611,NaN,North Vancouver Second Narrows,T006,2827,67297,574,574,North Vancouver Second Narrows,EXISTS
3,"Attribution, Concatenate and delete",12563,NA -> Rocky Point Park,Port Moody Rocky Point Park,T009,5658,151939,1148,1148,Port Moody Rocky Point Park,EXISTS


In [17]:
from sqlalchemy import text

# 1️⃣ Query to select old observations that need moving
SQL_TO_MOVE = text("""
SELECT o_old.history_id AS old_hist_id,
       o_old.obs_time,
       o_old.vars_id
FROM obs_raw o_old
JOIN meta_history h_old ON o_old.history_id = h_old.history_id
JOIN meta_station s_old ON h_old.station_id = s_old.station_id
WHERE s_old.native_id = :old_id

EXCEPT

SELECT o_new.history_id AS old_hist_id,
       o_new.obs_time,
       o_new.vars_id
FROM obs_raw o_new
JOIN meta_history h_new ON o_new.history_id = h_new.history_id
JOIN meta_station s_new ON h_new.station_id = s_new.station_id
WHERE s_new.native_id = :new_id
""")

# 2️⃣ Get the new station's history_id (assume we use latest)
SQL_NEW_HISTORY = text("""
SELECT history_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id = :new_id
""")

def move_station_obs_fast(old_id, new_id, engine):
    with engine.begin() as conn:
        # get new history_id (latest)
        new_hist_id = conn.execute(SQL_NEW_HISTORY, {"new_id": new_id}).scalar()
        if new_hist_id is None:
            print(f"New station '{new_id}' has no history, skipping.")
            return 0

        # update all in one query
        n_move = conn.execute(
            text("""
                WITH updated AS (
                    UPDATE obs_raw o
                    SET history_id = :new_hist_id
                    FROM meta_history h_old
                    JOIN meta_station s_old ON h_old.station_id = s_old.station_id
                    WHERE o.history_id = h_old.history_id
                      AND s_old.native_id = :old_id
                      AND NOT EXISTS (
                          SELECT 1
                          FROM obs_raw o_new
                          JOIN meta_history h_new ON o_new.history_id = h_new.history_id
                          JOIN meta_station s_new ON h_new.station_id = s_new.station_id
                          WHERE s_new.native_id = :new_id
                            AND o_new.obs_time = o.obs_time
                            AND o_new.vars_id = o.vars_id
                      )
                    RETURNING o.*
                )
                SELECT COUNT(*) FROM updated;
            """),
            {"old_id": old_id, "new_id": new_id, "new_hist_id": new_hist_id}
        ).scalar()

        print(f"Moved {n_move} rows from '{old_id}' to '{new_id}'")
        return n_move


# 4️⃣ Loop through your dataframe
results = []

for i, row in df_attr_concat_native.iterrows():  # preview first 2 rows
    old_id = row['old_native_id']
    new_id = row['new_native_id']
    print(f"[{i+1}/{len(df_attr_concat_native)}] Processing old='{old_id}' -> new='{new_id}'")
    n = move_station_obs_fast(old_id, new_id, engine)
    results.append(n)

# df_attr_concat_native['n_moved'] = results
print("All done!")

[1/4] Processing old='Peace Valley Attachie Flat Upper Terrace' -> new='AFU'
Moved 2819 rows from 'Peace Valley Attachie Flat Upper Terrace' to 'AFU'
[2/4] Processing old='Burnaby Kensington Park' -> new='T004'
Moved 2151 rows from 'Burnaby Kensington Park' to 'T004'
[3/4] Processing old='North Vancouver Second Narrows' -> new='T006'
Moved 2253 rows from 'North Vancouver Second Narrows' to 'T006'
[4/4] Processing old='Port Moody Rocky Point Park' -> new='T009'
Moved 4510 rows from 'Port Moody Rocky Point Park' to 'T009'
All done!


### The left four, using station_id to concat

In [58]:
df_attr_concat_stationid = df_attr_concat.loc[4:, ['ISSUE', 'Station ID', 'old_native_id', 'new_native_id']]

df_attr_concat_stationid['new_station_id'] = (
    df_attr_concat_stationid['ISSUE']
    .str.extract(r'(\d+)$')
    .astype(int)   # optional: keeps NaN where no match
)

df_attr_concat_stationid = df_attr_concat_stationid.rename(columns={
    'Station ID': 'old_station_id',
})

df_attr_concat_stationid


,ISSUE,old_station_id,old_native_id,new_native_id,new_station_id
4,"Attribution, ID, Name, Location, Concatenate w...",12582,Vancouver Clark Drive,T050,12949
5,"Attribution, ID, Name, Location, Concatenate w...",12949,E316370,T050,12949
6,"Attribution,Concatenate w/12536",2641,M110517,T012,12536
7,"Attribution,Concatenate w/12536",2637,M110516,T012,12536


In [59]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE m_old.station_id = %s
       AND m_new.station_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def count_station_stats(
    old_station_id,
    new_station_id,
    engine
):
    params = (
        # n_old
        old_station_id,

        # n_new
        new_station_id,

        # n_overlap
        old_station_id,
        new_station_id,

        # n_overlap_same_datum
        old_station_id,
        new_station_id,
    )

    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]

stats = df_attr_concat_stationid.apply(
    lambda r: count_station_stats(
        r["old_station_id"],
        r["new_station_id"],
        engine
    ),
    axis=1
)

df_attr_concat_stationid[[
    "n_old",
    "n_new",
    "n_overlap",
    "n_overlap_same_datum"
]] = stats

In [60]:
query = """
SELECT native_id
FROM meta_station
WHERE station_id = %s
"""

def get_new_native_id_from_s(concat_station_id, engine):
    # handle <NA> safely
    if pd.isna(concat_station_id):
        return None

    df = pd.read_sql(
        query,
        engine,
        params=(int(concat_station_id),)
    )

    if df.empty:
        return None

    return df.iloc[0]["native_id"]  # TEXT → string

df_attr_concat_stationid["native_id_from_db"] = df_attr_concat_stationid.apply(
    lambda row: get_new_native_id_from_s(
        row["new_station_id"],
        engine
    ),
    axis=1
)

In [61]:
df_attr_concat_stationid

,ISSUE,old_station_id,old_native_id,new_native_id,new_station_id,n_old,n_new,n_overlap,n_overlap_same_datum,native_id_from_db
4,"Attribution, ID, Name, Location, Concatenate w...",12582,Vancouver Clark Drive,T050,12949,1148,103270,1148,1148,E316370
5,"Attribution, ID, Name, Location, Concatenate w...",12949,E316370,T050,12949,103270,103270,103270,103270,E316370
6,"Attribution,Concatenate w/12536",2641,M110517,T012,12536,0,160146,0,0,T012
7,"Attribution,Concatenate w/12536",2637,M110516,T012,12536,0,160146,0,0,T012


In [62]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.station_id = :station_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in df_attr_concat_stationid.iterrows():
        station_id = row['new_station_id']

        exists = conn.execute(
            exists_sql,
            {"station_id": station_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(df_attr_concat_stationid)}] "
            f"new_station_id={station_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
df_attr_concat_stationid['new_station_status'] = exists_list

[  5/4] new_station_id=12949           | → EXISTS
[  6/4] new_station_id=12949           | → EXISTS
[  7/4] new_station_id=12536           | → EXISTS
[  8/4] new_station_id=12536           | → EXISTS


### Concat based on station_id

In [68]:
from sqlalchemy import text

SQL_NEW_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
WHERE h.station_id = :new_station_id
ORDER BY h.history_id DESC
LIMIT 1
""")

SQL_MOVE = text("""
WITH updated AS (
    UPDATE obs_raw o
    SET history_id = :new_hist_id
    FROM meta_history h_old
    WHERE o.history_id = h_old.history_id
      AND h_old.station_id = :old_station_id

      AND NOT EXISTS (
          SELECT 1
          FROM obs_raw o_new
          JOIN meta_history h_new ON o_new.history_id = h_new.history_id
          WHERE h_new.station_id = :new_station_id
            AND o_new.obs_time = o.obs_time
            AND o_new.vars_id  = o.vars_id
      )
    RETURNING 1
)
SELECT COUNT(*) FROM updated;
""")

def move_station_obs_fast(
    old_station_id,
    new_station_id,
    engine
):
    with engine.begin() as conn:
        new_hist_id = conn.execute(
            SQL_NEW_HISTORY,
            {"new_station_id": new_station_id}
        ).scalar()

        if new_hist_id is None:
            print(f"New station {new_station_id} has no history, skipping.")
            return 0

        n_move = conn.execute(
            SQL_MOVE,
            {
                "old_station_id": old_station_id,
                "new_station_id": new_station_id,
                "new_hist_id": new_hist_id,
            }
        ).scalar()

        print(
            f"Moved {n_move} rows "
            f"from station {old_station_id} → {new_station_id}"
        )

        return n_move

results = []

for _, row in df_attr_concat_stationid.iterrows():
    n = move_station_obs_fast(
        row["old_station_id"],
        row["new_station_id"],
        engine
    )
    results.append(n)

df_attr_concat_stationid["n_moved"] = results

Moved 0 rows from station 12582 → 12949
Moved 0 rows from station 12949 → 12949
Moved 0 rows from station 2641 → 12536
Moved 0 rows from station 2637 → 12536


### Delete old stations

In [69]:
### First delete the three stations with issue marked as delete

df_attr_concat

df_attr_concat_del = df_attr_concat[
    df_attr_concat["ISSUE"].str.contains("delete", case=False, na=False)
]

,ISSUE,Station ID,Unique Names,old_native_id,new_native_id,n_old,n_new,n_overlap,n_overlap_same_datum,old_native_id_db,new_station_status
0,"Attribution, Concatenate",12559,NA -> Peace Valley Attachie Flat Upper Terrace,Peace Valley Attachie Flat Upper Terrace,AFU,0,479390,0,0,Peace Valley Attachie Flat Upper Terrace,EXISTS
1,"Attribution, Concatenate and delete",12592,NaN,Burnaby Kensington Park,T004,574,91903,574,574,Burnaby Kensington Park,EXISTS
2,"Attribution, Concatenate and delete",12611,NaN,North Vancouver Second Narrows,T006,574,69550,574,574,North Vancouver Second Narrows,EXISTS
3,"Attribution, Concatenate and delete",12563,NA -> Rocky Point Park,Port Moody Rocky Point Park,T009,1148,156449,1148,1148,Port Moody Rocky Point Park,EXISTS
4,"Attribution, ID, Name, Location, Concatenate w...",12582,NA -> Vancouver Clark Drive,Vancouver Clark Drive,T050,1148,0,0,0,Vancouver Clark Drive,NOT EXISTS
5,"Attribution, ID, Name, Location, Concatenate w...",12949,Vancouver Clark Drive,E316370,T050,103270,0,0,0,E316370,NOT EXISTS
6,"Attribution,Concatenate w/12536",2641,Chilliwack Airport,M110517,T012,0,160146,0,0,M110517,EXISTS
7,"Attribution,Concatenate w/12536",2637,Chilliwack Airport Met,M110516,T012,0,160146,0,0,M110516,EXISTS


In [90]:
df_attr_concat_del = df_attr_concat_del.rename(columns={'Station ID': 'old_station_id'})

station_pairs = (
    df_attr_concat_del[["old_station_id", "old_native_id"]]
    .dropna()
    .assign(
        old_station_id=lambda d: d["old_station_id"].astype(int),
        old_native_id=lambda d: d["old_native_id"].astype(str),
    )
    .drop_duplicates()
    .to_records(index=False)
)

station_pairs

rec.array([(12592, 'Burnaby Kensington Park'),
           (12611, 'North Vancouver Second Narrows'),
           (12563, 'Port Moody Rocky Point Park')],
          dtype=[('old_station_id', '<i8'), ('old_native_id', 'O')])

In [91]:
delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o
JOIN meta_history h ON o.history_id = h.history_id
JOIN meta_station s ON h.station_id = s.station_id
WHERE fr.obs_raw_id = o.obs_raw_id
  AND s.station_id = :station_id
  AND s.native_id  = :native_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE o.history_id = h.history_id
  AND s.station_id = :station_id
  AND s.native_id  = :native_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE dv.history_id = h.history_id
  AND s.station_id = :station_id
  AND s.native_id  = :native_id
""")

delete_history = sa.text("""
DELETE FROM meta_history h
USING meta_station s
WHERE h.station_id = s.station_id
  AND s.station_id = :station_id
  AND s.native_id  = :native_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
  AND native_id  = :native_id
""")

with engine.begin() as conn:
    for station_id, native_id in station_pairs:

        station_id = int(station_id)     # convert np.int64 → int
        native_id  = str(native_id)      # (safe, optional)

        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": station_id, "native_id": native_id}
        )

        res_obs = conn.execute(
            delete_obs,
            {"station_id": station_id, "native_id": native_id}
        )

        res_dv = conn.execute(
            delete_obs_derived,
            {"station_id": station_id, "native_id": native_id}
        )

        res_hist = conn.execute(
            delete_history,
            {"station_id": station_id, "native_id": native_id}
        )

        res_sta = conn.execute(
            delete_station,
            {"station_id": station_id, "native_id": native_id}
        )

        print(
            f"Station {station_id}/{native_id}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Station 12592/Burnaby Kensington Park: flags=0, obs_raw=574, obs_derived=0, meta_history=1, meta_station=1
Station 12611/North Vancouver Second Narrows: flags=0, obs_raw=574, obs_derived=0, meta_history=1, meta_station=1
Station 12563/Port Moody Rocky Point Park: flags=0, obs_raw=1148, obs_derived=0, meta_history=1, meta_station=1
